# Comparative Study of Community Detection in Complex Networks
**Author:** Kundan Kammula  
**Course:** Project 4 — Network Science & Graph Analytics  

This interactive notebook demonstrates the core capabilities and findings of our empirical evaluation of five community detection algorithms:
1. **Louvain** (Heuristic modularity maximization)
2. **Greedy Modularity / CNM** (Agglomerative bottom-up optimization)
3. **Label Propagation / LPA** (Asynchronous majority voting)
4. **Spectral Clustering** (Normalized Laplacian embedding + $k$-Means)
5. **Girvan-Newman** (Divisive edge-betweenness centrality removal)

In [ ]:
import os
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import cd_lib as C

# Plot styling configuration
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10
})
print("Loaded cd_lib core module successfully!")

## 1. Zachary's Karate Club Ground-Truth Validation (E1)
Evaluating algorithm partitions against the historic 2-faction ground-truth split.

In [ ]:
g_karate, truth_karate = C.load_karate()
results = []
partitions = {}

for name in C.ALGORITHMS:
    kw = {"restarts": 20} if name == "Louvain" else {}
    comms, runtime = C.run(name, g_karate, **kw)
    metrics = C.evaluate(g_karate, comms, truth_karate)
    metrics["algorithm"] = name
    metrics["time_s"] = runtime
    results.append(metrics)
    partitions[name] = comms

df_karate = pd.DataFrame(results).set_index("algorithm")
df_karate[["n_communities", "modularity", "coverage", "conductance", "nmi", "ari", "time_s"]].round(4)

## 2. Visualizing Karate Club Partitions
Compare the ground-truth faction split against partitions discovered by Louvain and Spectral Clustering.

In [ ]:
pos = nx.spring_layout(g_karate, seed=7, k=0.55)
palette = plt.cm.tab10.colors

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Ground Truth
nx.draw_networkx_nodes(g_karate, pos, ax=axes[0], node_color=[palette[t] for t in truth_karate], node_size=200)
nx.draw_networkx_edges(g_karate, pos, ax=axes[0], alpha=0.3)
nx.draw_networkx_labels(g_karate, pos, ax=axes[0], font_size=7, font_color='white')
axes[0].set_title(f"Ground Truth (2 Factions)\nModularity Q={df_karate.loc['Louvain', 'modularity']:.3f}")
axes[0].axis('off')

# 2. Louvain
louvain_labs = C.labels_from(g_karate, partitions['Louvain'])
nx.draw_networkx_nodes(g_karate, pos, ax=axes[1], node_color=[palette[l] for l in louvain_labs], node_size=200)
nx.draw_networkx_edges(g_karate, pos, ax=axes[1], alpha=0.3)
nx.draw_networkx_labels(g_karate, pos, ax=axes[1], font_size=7, font_color='white')
axes[1].set_title(f"Louvain (4 Sub-cliques)\nModularity Q={df_karate.loc['Louvain', 'modularity']:.3f} | NMI={df_karate.loc['Louvain', 'nmi']:.3f}")
axes[1].axis('off')

# 3. Spectral
spectral_labs = C.labels_from(g_karate, partitions['Spectral'])
nx.draw_networkx_nodes(g_karate, pos, ax=axes[2], node_color=[palette[l] for l in spectral_labs], node_size=200)
nx.draw_networkx_edges(g_karate, pos, ax=axes[2], alpha=0.3)
nx.draw_networkx_labels(g_karate, pos, ax=axes[2], font_size=7, font_color='white')
axes[2].set_title(f"Spectral Clustering (4 Clusters)\nModularity Q={df_karate.loc['Spectral', 'modularity']:.3f} | NMI={df_karate.loc['Spectral', 'nmi']:.3f}")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 3. Semantic Story Arcs in *Les Misérables* (E2)
Using Weighted Louvain to discover narrative subplots in Victor Hugo's co-appearance network.

In [ ]:
g_lesmis = C.load_lesmis()
comms_lesmis, _ = C.run("Louvain", g_lesmis, weight="weight", restarts=20)
comms_lesmis = sorted(comms_lesmis, key=len, reverse=True)

deg = dict(g_lesmis.degree(weight="weight"))
story_arcs = []
for idx, c in enumerate(comms_lesmis, start=1):
    lead_chars = sorted(c, key=lambda v: deg[v], reverse=True)[:5]
    story_arcs.append({
        "Community": f"Arc {idx}",
        "Size": len(c),
        "Lead Characters": ", ".join(lead_chars)
    })

pd.DataFrame(story_arcs)

## 4. Cartographic Node Roles (Guimerà & Amaral Framework) (E6)
Mapping nodes by **Within-Module Degree z-score ($z_i$)** (hub status) and **Participation Coefficient ($P_i$)** (bridging connectivity).

In [ ]:
zp = C.role_metrics(g_karate, partitions['Louvain'])
cen = C.centralities(g_karate)
lab_louvain = C.labels_from(g_karate, partitions['Louvain'])

df_roles = pd.DataFrame(cen)
df_roles["z_within"] = [zp[v][0] for v in g_karate]
df_roles["participation"] = [zp[v][1] for v in g_karate]
df_roles["community"] = [l + 1 for l in lab_louvain]
df_roles["role"] = [C.classify_role(df_roles.loc[v, 'z_within'], df_roles.loc[v, 'participation']) for v in g_karate]

print("=== Top Nodes by Betweenness Centrality ===")
display(df_roles.sort_values("betweenness", ascending=False).head(5).round(3))

print("\n=== Functional Role Distribution ===")
print(df_roles["role"].value_counts())

## 5. Statistical Significance via Degree-Preserving Null Models (E7)
Testing observed modularity against randomized configuration graphs with identical degree sequences.

In [ ]:
q_real_karate = df_karate.loc["Louvain", "modularity"]
null_qs_karate = C.null_modularity(g_karate, "Louvain", n_samples=30, restarts=5)

z_score = (q_real_karate - null_qs_karate.mean()) / null_qs_karate.std()

plt.figure(figsize=(7, 4))
plt.hist(null_qs_karate, bins=10, color='#cccccc', edgecolor='white', label='Randomized Null (30 Rewirings)')
plt.axvline(q_real_karate, color='#d62728', lw=2.5, label=f'Real Network Q = {q_real_karate:.3f}')
plt.title(f"Null Model Significance: Karate Club (z-score = {z_score:.2f})")
plt.xlabel("Louvain Modularity (Q)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Karate Club Real Modularity: {q_real_karate:.4f}")
print(f"Null Model Mean Modularity: {null_qs_karate.mean():.4f} +/- {null_qs_karate.std():.4f}")
print(f"Computed z-score: {z_score:.2f} (Statistically authentic!)")